In [23]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [24]:
df = pd.read_csv("../data/processed/bank-full_cleaned.csv")
df.head()

,job,marital,education,default,balance,contact,day,month,campaign,pdays,previous,poutcome,aceptacion,contactado_antes,n_creditos,en_sobregiro,rango
0,management,married,tertiary,no,2143,unknown,5,may,1,NaN,0,unknown,0,0,1,0,>50
1,technician,single,secondary,no,29,unknown,5,may,1,NaN,0,unknown,0,0,1,0,31-50
2,entrepreneur,married,secondary,no,2,unknown,5,may,1,NaN,0,unknown,0,0,2,0,31-50
3,blue-collar,married,unknown,no,1506,unknown,5,may,1,NaN,0,unknown,0,0,1,0,31-50
4,unknown,single,unknown,no,1,unknown,5,may,1,NaN,0,unknown,0,0,0,0,31-50


In [25]:
df['pdays'].isna().sum()

np.int64(36954)

In [26]:
df.shape

(45211, 17)

In [27]:
X = df.drop(columns='aceptacion')
y = df['aceptacion']

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, stratify=y, random_state=42)

In [29]:
X_train.shape

(36168, 16)

In [30]:
y_train.shape

(36168,)

In [31]:
X_test.shape

(9043, 16)

In [32]:
y_test.shape

(9043,)

In [33]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 36168 entries, 24001 to 44229
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   job               36168 non-null  str    
 1   marital           36168 non-null  str    
 2   education         36168 non-null  str    
 3   default           36168 non-null  str    
 4   balance           36168 non-null  int64  
 5   contact           36168 non-null  str    
 6   day               36168 non-null  int64  
 7   month             36168 non-null  str    
 8   campaign          36168 non-null  int64  
 9   pdays             6584 non-null   float64
 10  previous          36168 non-null  int64  
 11  poutcome          36168 non-null  str    
 12  contactado_antes  36168 non-null  int64  
 13  n_creditos        36168 non-null  int64  
 14  en_sobregiro      36168 non-null  int64  
 15  rango             36168 non-null  str    
dtypes: float64(1), int64(7), str(8)
memory usage: 4.7 MB

In [34]:
y_train.mean()

np.float64(0.11698186241981863)

In [16]:
y_test.mean()

np.float64(0.11699657193409267)

In [35]:
num_cols = ['balance', 'campaign', 'previous', 'day']
cat_cols = ['job', 'marital', 'education', 'contact', 'poutcome', 'month', 'n_creditos', 'rango', 'default']

pre_dias = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("dias", pre_dias, ["pdays"]),
    ("cat", OneHotEncoder(handle_unknown="ignore",drop="if_binary"), cat_cols),
    ("bin", "passthrough", ['contactado_antes','en_sobregiro']),
])

X_train_pre = preprocesador.fit_transform(X_train)   # fit: aprende SOLO de train
X_test_pre = preprocesador.transform(X_test)         # transform: aplica lo aprendido, no recalcula

print("train:", X_train_pre.shape, "| test:", X_test_pre.shape)

train: (36168, 52) | test: (9043, 52)


In [38]:
pipe_base = Pipeline([
    ("pre", preprocesador),
    ("modelo", LogisticRegression(max_iter=1000)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [39]:
print("Exactitud de 'siempre digo no':", round(1 - y_train.mean(), 3))

proba_cv = cross_val_predict(pipe_base, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
pred_cv = (proba_cv >= 0.5).astype(int)

print("\nMatriz de confusión (filas = real, columnas = predicho):")
print(confusion_matrix(y_train, pred_cv))
print(classification_report(y_train, pred_cv, digits=3))
print("AUC:", round(roc_auc_score(y_train, proba_cv), 3))

Exactitud de 'siempre digo no': 0.883

Matriz de confusión (filas = real, columnas = predicho):
[[31521   416]
 [ 3489   742]]
              precision    recall  f1-score   support

           0      0.900     0.987     0.942     31937
           1      0.641     0.175     0.275      4231

    accuracy                          0.892     36168
   macro avg      0.771     0.581     0.609     36168
weighted avg      0.870     0.892     0.864     36168

AUC: 0.765


### Modelo base: regresión logística

- Datos: `data/processed`, split 80/20 estratificado (`random_state=42`).
- Preprocesamiento dentro del `Pipeline`, ajustado solo con train: one-hot para categóricas, `rango` de edad, `month` y `n_creditos`; escalado de numéricas; `pdays` con mediana y escalado.
- **Decisión `pdays`:** imputar con la mediana y dejar el flag `contactado_antes`. El relleno es solo un marcador para que la regresión no falle; la información de "no contactado" la lleva el flag.
- Resultados (validación cruzada de 5 folds sobre train, umbral 0,5): exactitud 0,892 (frente a 0,883 de "siempre digo no"), recall 0,175, precisión 0,641, AUC 0,765.
- **Lectura:** detecta ~1,7 de cada 10 clientes que sí contratan. Con el corte en 0,5 es muy cauteloso para decir "sí". Para el negocio, perder un "sí" cuesta más que una llamada de más.
- Pendiente: probar umbrales más bajos y `class_weight`.